# Notebook 17 benchmark definitions

Purpose: audit a fixed positive-feature content kernel on frozen, training-only ViT representations, and measure complete JEPA steps with the same optimized image pipeline. This file exports definitions only; the front-door Notebook 17 launches the bounded experiment.

Inputs: source-bound Notebook 15 ViT checkpoint, selected training IDs, ranks 32/64/128, exact 3 × 3 local correction. Outputs: operator measurements, checkpoint manifests and complete-step speed/memory gate. Expected runtime: bounded by the immutable two-hour study deadline; each compression case has a 180-second subprocess watchdog. Checkpoints live in `checkpoints/<run_id>/compression/`; resume through the front-door launch cell with the same run ID and unchanged source.

The finite-feature dense reference establishes implementation correctness on small cases. Errors against exact SDPA at realistic token counts are empirical fidelity diagnostics, not a theorem about all inputs or image accuracy. Promotion requires the declared fidelity guards and a measured full-JEPA engineering win.

In [ ]:
"""Source-bound fixed-feature compression and complete JEPA timing."""
from pathlib import Path
import copy,gc,hashlib,json,math,os,signal,statistics,subprocess,sys,tempfile,time,types
import nbformat,numpy as np,torch
ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()

def _load(root,relative,name):
    path=Path(root)/relative
    if name in sys.modules:
        result=sys.modules[name]
        if Path(result.__file__).resolve()!=path.resolve():raise ValueError('Notebook module identity changed')
        return result
    result=types.ModuleType(name);result.__file__=str(path);sys.modules[name]=result
    try:
        for cell in nbformat.read(path,4).cells:
            if cell.cell_type=='code'and'ahr-export'in cell.metadata.get('tags',[]):exec(compile(cell.source,str(path),'exec'),result.__dict__)
    except BaseException:sys.modules.pop(name,None);raise
    return result

def _common(root=ROOT):return _load(root,'experiments/17_fixed_kernel/common.ipynb','fmm17_common')
def _operator(root=ROOT):return _load(root,'experiments/17_fixed_kernel/operator.ipynb','fmm17_operator')
def _pipeline(root=ROOT):return _load(root,'experiments/17_fixed_kernel/pipeline.ipynb','fmm17_pipeline')
def _runner(root=ROOT):return _load(root,'experiments/17_fixed_kernel/runner.ipynb','fmm17_runner')
def _canonical(value):return json.loads(json.dumps(value,allow_nan=False))
def _digest(value):return hashlib.sha256(json.dumps(_canonical(value),sort_keys=True,separators=(',',':')).encode()).hexdigest()
def _record(path):
    path=Path(path);return {'path':str(path.resolve()),'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
def _atomic_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+'.tmp-'+str(os.getpid()))
    with temporary.open('w')as handle:json.dump(_canonical(value),handle,indent=2,sort_keys=True);handle.flush();os.fsync(handle.fileno())
    os.replace(temporary,path)
def _read(path):return json.loads(Path(path).read_text())
def _immutable(path,value):
    value=_canonical(value);path=Path(path)
    if path.exists():
        if _read(path)!=value:raise ValueError('Immutable benchmark identity differs: '+str(path))
    else:_atomic_json(path,value)
    return value

def _deadline(deadline,required=0.):
    if deadline is not None and time.time()+float(required)>=float(deadline):raise TimeoutError('Original study deadline exhausted; UNKNOWN, never reset')

def _rng_state(device='cpu'):
    import random
    result={'python':random.getstate(),'numpy':np.random.get_state(),'torch_cpu':torch.get_rng_state()}
    if str(device).startswith('cuda'):result['torch_cuda']=torch.cuda.get_rng_state_all()
    return result

def _ancestor_pids():
    allowed={os.getpid()};pid=os.getpid()
    for _ in range(32):
        try:
            values=Path('/proc/'+str(pid)+'/stat').read_text().rsplit(')',1)[1].split();parent=int(values[1])
        except (OSError,ValueError,IndexError):break
        if parent<=1 or parent in allowed:break
        allowed.add(parent);pid=parent
    return sorted(allowed)

def _pmon_sample(index,allowed):
    result=subprocess.run(['nvidia-smi','pmon','-i',str(index),'-s','u','-c','1'],capture_output=True,text=True,timeout=5,check=True)
    rows=[];active=[]
    for line in result.stdout.splitlines():
        if not line.strip()or line.lstrip().startswith('#'):continue
        fields=line.split()
        if len(fields)<5:raise RuntimeError('UNKNOWN_GPU_TELEMETRY: pmon row cannot be parsed')
        if fields[1]=='-':continue
        try:pid=int(fields[1]);sm=None if fields[3]=='-'else float(fields[3]);gpu=int(fields[0])
        except ValueError:raise RuntimeError('UNKNOWN_GPU_TELEMETRY: invalid PID/SM observation')
        if gpu!=index:raise RuntimeError('UNKNOWN_GPU_TELEMETRY: pmon device differs')
        row={'pid':pid,'sm_percent':sm,'sm_dash_observed':fields[3]=='-','memory_activity_percent':None if fields[4]=='-'else float(fields[4]),'allowed_current_or_ancestor':pid in allowed}
        rows.append(row)
        if pid not in allowed and sm is not None and sm>0:active.append(row)
    return {'observed_unix':time.time(),'rows':rows,'foreign_active':active,'raw':result.stdout}

def _exclusive_gpu(device):
    """Conditional observed compute availability; resident idle contexts are not activity.

    Current process/ancestors may have idle CUDA contexts from checkpoint RNG.
    Foreign contexts require three observed idle SM samples. Any active foreign
    SM or missing telemetry refuses the timing. This is not an absolute isolation
    claim; a background observer additionally watches the measured window.
    """
    if not str(device).startswith('cuda'):return {'status':'CPU_VALIDATION_ONLY','eligible':True}
    index=torch.device(device).index or 0;allowed=_ancestor_pids()
    processes=subprocess.run(['nvidia-smi','--query-compute-apps=pid,gpu_uuid','--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=5,check=True)
    devices=subprocess.run(['nvidia-smi','--query-gpu=index,uuid,utilization.gpu,memory.used','--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=5,check=True)
    row=next(line.split(',')for line in devices.stdout.splitlines()if int(line.split(',')[0])==index);uuid=row[1].strip()
    foreign=[int(line.split(',')[0])for line in processes.stdout.splitlines()if line.strip()and line.split(',')[1].strip()==uuid and int(line.split(',')[0])not in allowed]
    samples=[_pmon_sample(index,allowed)for _ in range(3)]
    if any(s['foreign_active']for s in samples):raise RuntimeError('UNKNOWN_FOREIGN_GPU_COMPUTE: observed active foreign SM on benchmark device')
    observed={r['pid']for sample in samples for r in sample['rows']}
    if not set(foreign).issubset(observed):raise RuntimeError('UNKNOWN_GPU_TELEMETRY: resident foreign context has no pmon idle observation')
    return {'status':'OBSERVED_NO_FOREIGN_COMPUTE','eligible':True,'gpu_index':index,'gpu_uuid':uuid,'pid':os.getpid(),'allowed_current_and_ancestor_pids':allowed,
      'foreign_idle_context_pids':foreign,'gpu_utilization_percent':float(row[2].strip()),'gpu_memory_used_mib':float(row[3].strip()),'pmon_samples':samples,'checked_unix':time.time(),
      'scope':'Three pre/post point observations show no foreign SM activity. Idle resident contexts allowed; background pmon watches timed windows. Conditional observation, not absolute GPU exclusivity; other GPU/CPU workload can still affect host throughput.'}

class _ComputeObserver:
    _active=set()
    def __init__(self,device):
        self.device=str(device);self.samples=[];self.errors=[];self.thread=None;self.initial=None
    def start(self):
        if not self.device.startswith('cuda'):return self
        import threading
        self.initial=_exclusive_gpu(self.device);self.allowed=self.initial['allowed_current_and_ancestor_pids'];self.index=self.initial['gpu_index'];self.stop=threading.Event()
        def observe():
            while not self.stop.is_set():
                try:self.samples.append(_pmon_sample(self.index,self.allowed))
                except BaseException as error:self.errors.append(repr(error));break
                self.stop.wait(.25)
        self.thread=threading.Thread(target=observe,daemon=True);self.thread.start();self._active.add(self);return self
    def abort(self):
        if self.thread is not None:
            self.stop.set();self.thread.join(timeout=6);self._active.discard(self)
            if self.thread.is_alive():raise RuntimeError('UNKNOWN_GPU_TELEMETRY: monitor failed to stop')
    def finish(self):
        if self.thread is None:return {'status':'CPU_VALIDATION_ONLY','eligible':True,'background_samples':[]}
        self.abort()
        if self.errors or any(s['foreign_active']for s in self.samples):raise RuntimeError('UNKNOWN_FOREIGN_GPU_COMPUTE_OR_TELEMETRY: timed-window monitoring failed '+repr(self.errors))
        final=_exclusive_gpu(self.device)
        return {'status':'OBSERVED_NO_FOREIGN_COMPUTE','eligible':True,'before':self.initial,'after':final,'background_samples':self.samples,
          'scope':'No foreign SM activity was observed before, during or after this timed window. Sampling cannot prove absence between observations.'}


In [ ]:
def _gradient_errors(reference,candidate):
    errors={name:float((candidate['gradients'][name]-value).norm()/value.norm().clamp_min(1e-8))for name,value in reference['gradients'].items()}
    numerator=sum(float((candidate['gradients'][name]-value).square().sum())for name,value in reference['gradients'].items())
    denominator=sum(float(value.square().sum())for value in reference['gradients'].values())
    return {'relative_output_l2':float((candidate['output']-reference['output']).norm()/reference['output'].norm().clamp_min(1e-12)),
      'relative_gradient_l2':errors,'max_relative_gradient_l2':max(errors.values(),default=0.),'aggregate_relative_gradient_l2':math.sqrt(numerator/max(denominator,1e-16))}

def measured_operator(mixer,tokens,indices,valid,grid,implementation,options,repeats=3,device='cuda:0',dtype=torch.float32):
    """Whole learned QKV + mixer + projection forward and every shared parameter gradient."""
    operator=_operator();current=copy.deepcopy(mixer).to(device=device,dtype=dtype);current.requires_grad_(True)
    x0=tokens.detach().to(device=device,dtype=dtype);indices=indices.to(device);valid=valid.to(device)
    B,N,D=x0.shape;heads=current.heads;head_dim=D//heads;grid=(int(grid),int(grid))if isinstance(grid,int)else tuple(grid)
    feature=operator.make_orthogonal_features(heads,head_dim,int(options['rank']),int(options['feature_seed']),dtype=dtype).to(device)
    parameter_items=[(name,p)for name,p in current.named_parameters()if p.requires_grad]
    names=[name for name,p in parameter_items];parameters=tuple(p for name,p in parameter_items)
    observations=[];payload=None;last_stats={};observer=_ComputeObserver(device).start()
    def forward(x,instrument=False):
        safe=torch.where(valid[...,None],x,torch.zeros_like(x));q,k,v=current.qkv(safe).reshape(B,N,3,heads,head_dim).permute(2,0,3,1,4).unbind(0)
        if implementation=='sdpa':
            values=operator.exact_softmax_attention(q,k,v,valid,indices=indices,grid=grid)
            stats={'algorithm':'torch.scaled_dot_product_attention observed runtime dispatch','rank':None,'far_matrix_constructed':False}
        else:
            function=operator.dense_fixed_kernel_reference if implementation=='finite_dense'else operator.fixed_kernel_attention
            if implementation=='finite_dense':
                values=function(q,k,v,valid,feature_matrix=feature,indices=indices,grid=grid,local_radius=int(options.get('local_radius',1)));stats={'algorithm':'Independent dense finite-feature reference, correctness only'}
            else:
                result=function(q,k,v,valid,feature_matrix=feature,indices=indices,grid=grid,local_radius=int(options.get('local_radius',1)),return_stats=instrument)
                values,stats=result if instrument else(result,{})
        output=current.projection(values.transpose(1,2).reshape_as(x))*valid[...,None]
        return output,stats
    for repeat in range(int(repeats)+1):
        x=x0.clone().requires_grad_();gc.collect()
        if str(device).startswith('cuda'):
            torch.cuda.synchronize(device);torch.cuda.reset_peak_memory_stats(device)
            e0=torch.cuda.Event(enable_timing=True);e1=torch.cuda.Event(enable_timing=True);e2=torch.cuda.Event(enable_timing=True);e0.record()
        begin=time.perf_counter();output,stats=forward(x)
        if str(device).startswith('cuda'):e1.record();torch.cuda.synchronize(device)
        middle=time.perf_counter()
        # A fixed scalar objective exercises all channels; it is not a training-quality metric.
        weights=torch.linspace(.5,1.5,output.shape[-1],device=device,dtype=dtype)
        objective=(output.square()*weights).mean()+.01*(output*weights).mean()
        gradients=torch.autograd.grad(objective,(x,)+parameters,allow_unused=True)
        if str(device).startswith('cuda'):e2.record();torch.cuda.synchronize(device)
        end=time.perf_counter();finite=bool(torch.isfinite(output).all())and all(g is None or bool(torch.isfinite(g).all())for g in gradients)
        if not finite:raise FloatingPointError('Nonfinite operator output or gradient; promotion prohibited')
        payload={'output':output.detach().cpu(),'gradients':{'input':gradients[0].detach().cpu(),**{name:torch.zeros_like(parameter.detach().cpu())if gradient is None else gradient.detach().cpu()for name,parameter,gradient in zip(names,parameters,gradients[1:])}}}
        if repeat:
            observation={'forward_seconds':middle-begin,'backward_seconds':end-middle,'forward_backward_seconds':end-begin,
              'cuda_forward_seconds':e0.elapsed_time(e1)/1000 if str(device).startswith('cuda')else None,
              'cuda_backward_seconds':e1.elapsed_time(e2)/1000 if str(device).startswith('cuda')else None,
              'cuda_forward_backward_seconds':e0.elapsed_time(e2)/1000 if str(device).startswith('cuda')else None,
              'peak_cuda_allocated_bytes':torch.cuda.max_memory_allocated(device)if str(device).startswith('cuda')else None}
            observations.append(observation)
        last_stats=stats;del x,output,objective,gradients
    with torch.no_grad():_,last_stats=forward(x0,instrument=True)
    compute_observation=observer.finish()
    result={'status':'MEASURED','implementation':implementation,'dtype':str(dtype),'batch':B,'heads':heads,'tokens':N,'rank':int(options['rank']),
      'forward_backward_seconds':statistics.median(v['forward_backward_seconds']for v in observations),
      'cuda_forward_backward_seconds':statistics.median(v['cuda_forward_backward_seconds']for v in observations)if str(device).startswith('cuda')else None,
      'peak_cuda_allocated_bytes':max((v['peak_cuda_allocated_bytes']for v in observations if v['peak_cuda_allocated_bytes']is not None),default=None),
      'measurements':observations,'operator':_canonical(last_stats),'basis_sha256':operator.basis_sha256(feature),
      'all_shared_parameter_names':names,'observed_compute_availability':compute_observation,'mask_policy':'Same query/key padding validity; invalid outputs forced to zero','timing':'One warmup excluded; complete synchronized QKV/mixer/projection forward and autograd backward; CUDA events and wall measured separately.'}
    del current,feature;gc.collect()
    if str(device).startswith('cuda'):torch.cuda.empty_cache()
    return result,payload

def implementation_correctness(root=ROOT,device='cpu',rank=32):
    """Independent dense finite-feature reference, all parameters, padded and fully masked cases."""
    torch.set_num_threads(2);torch.manual_seed(171717);dtype=torch.float64 if device=='cpu'else torch.float32
    mixer=torch.nn.Module();mixer.heads=2;mixer.qkv=torch.nn.Linear(16,48,bias=True);mixer.projection=torch.nn.Linear(16,16,bias=True)
    options={'rank':int(rank),'feature_seed':170032,'local_radius':1,'spatial_mode':'disabled'}
    rows=[]
    for mask in('full','padding','empty'):
        tokens=torch.randn(2,25,16,dtype=dtype)*.1;indices=torch.arange(25).expand(2,-1).clone();valid=torch.ones(2,25,dtype=torch.bool)
        if mask=='padding':valid[:,::4]=False;indices[~valid]=-1
        elif mask=='empty':valid[:]=False;indices[:]=-1
        reference,ref=measured_operator(mixer,tokens,indices,valid,5,'finite_dense',options,repeats=1,device=device,dtype=dtype)
        measured,payload=measured_operator(mixer,tokens,indices,valid,5,'fixed',options,repeats=1,device=device,dtype=dtype)
        errors=_gradient_errors(ref,payload);tol=1e-9 if dtype==torch.float64 else 5e-5
        passed=errors['relative_output_l2']<=tol and errors['max_relative_gradient_l2']<=tol
        if mask=='empty':passed=passed and bool(payload['output'].eq(0).all())and all(bool(g.eq(0).all())for g in payload['gradients'].values())
        rows.append({'mask':mask,'passed':passed,'tolerance':tol,'errors':errors,'basis_sha256':measured['basis_sha256']})
    return {'status':'PASS'if all(v['passed']for v in rows)else'FAIL','passed':all(v['passed']for v in rows),'device':device,'dtype':str(dtype),'rank':int(rank),'rows':rows,
      'scope':'Implementation correctness against the same finite feature kernel; not approximation fidelity versus exact attention.'}


In [ ]:
def learned_tokens(model,bundle,config,ids,image_size,layer):
    from fmm_jepa.food101_data import EvaluationDataset
    changed=copy.deepcopy(config);changed['image_size']=int(image_size);data=EvaluationDataset(bundle['dataset'],ids,changed)
    images=torch.stack([data[i]['image']for i in range(len(ids))]);target=model.target.to(config['device']).eval();captured=[]
    handle=target.blocks[int(layer)].norm1.register_forward_hook(lambda module,args,out:captured.append(out.detach().clone()))
    try:
        with torch.no_grad():target(images.to(config['device']))
    finally:handle.remove()
    tokens=captured[-1].cpu();grid=int(image_size)//int(config['model']['patch_size']);indices=torch.arange(grid*grid).expand(len(ids),-1).clone()
    return tokens,indices,torch.ones_like(indices,dtype=torch.bool),(grid,grid),copy.deepcopy(target.blocks[int(layer)].mixer).cpu()

def run_compression_case(root,job_path):
    """Fresh subprocess entry point; reusable logic remains in this notebook."""
    root=Path(root);job=_read(job_path);common=_common(root);operator=_operator(root)
    _deadline(job.get('deadline'),1);device=job['device'];common.environment({'device':device});exclusive=_exclusive_gpu(device)
    options=job['options'];fixture=job.get('software_fixture',False)
    if job.get('correctness_only'):
        result=implementation_correctness(root,device,int(options['rank']));result['source_hashes']=job['source_hashes'];_atomic_json(job['result_path'],result);return result
    if fixture:
        torch.manual_seed(1717);mixer=torch.nn.Module();mixer.heads=2;mixer.qkv=torch.nn.Linear(16,48);mixer.projection=torch.nn.Linear(16,16)
        grid=5;tokens=torch.randn(2,25,16)*.05;indices=torch.arange(25).expand(2,-1);valid=torch.ones_like(indices,dtype=torch.bool);source={'role':'SYNTHETIC_SOFTWARE_VALIDATION'};sample_ids=[]
    else:
        model,config,source=common.learned_source(root,job.get('handoff'),variant='vit');config['device']=device;common.environment(config)
        bundle=common.data_bundle(root,Path(job['artifact'])/'source_data',config,verify_images=False)
        # Four random training members are registered independently of scores.
        rng=np.random.default_rng(2026171701);sample_ids=sorted(rng.choice(bundle['train_ids'],size=4,replace=False).tolist())
        image_size=int(math.isqrt(int(job['tokens'])))*int(config['model']['patch_size'])
        tokens,indices,valid,grid,mixer=learned_tokens(model,bundle,config,sample_ids,image_size,int(job['layer']))
        model.cpu();del model;gc.collect();torch.cuda.empty_cache()
    exact,reference=measured_operator(mixer,tokens,indices,valid,grid,'sdpa',options,repeats=1 if fixture else 3,device=device)
    _deadline(job.get('deadline'),1)
    measured,payload=measured_operator(mixer,tokens,indices,valid,grid,'fixed',options,repeats=1 if fixture else 3,device=device)
    errors=_gradient_errors(reference,payload)
    passed=errors['relative_output_l2']<=float(job.get('output_guard',.10))and errors['max_relative_gradient_l2']<=float(job.get('gradient_guard',.25))
    row={'key':job['key'],'status':'MEASURED','tokens':int(tokens.shape[1]),'layer':int(job.get('layer',0)),'rank':int(options['rank']),'sample_ids':sample_ids,'source':source,
      'measurements':{'sdpa':exact,'fixed':measured},'errors':errors,'fidelity_guard_pass':passed,
      'operator_speed_over_sdpa':exact['forward_backward_seconds']/measured['forward_backward_seconds'],'exclusive_gpu':exclusive,
      'role':'SYNTHETIC_SOFTWARE_VALIDATION'if fixture else'FROZEN_TRAINING_ONLY_LEARNED_TOKENS','rng':{'sampling_seed':2026171701,'feature_seed':int(options['feature_seed'])},
      'basis_sha256':measured['basis_sha256'],'basis_shape':[mixer.heads,int(options['rank']),tokens.shape[-1]//mixer.heads],
      'local_policy':'Exact raw-softmax 3x3 correction with original patch indices; positive finite-feature far summaries','finished_unix':time.time()}
    rng_path=Path(job['result_path']).with_suffix('.rng.pt');rng_path.parent.mkdir(parents=True,exist_ok=True);temporary=rng_path.with_suffix('.tmp')
    with temporary.open('wb')as handle:
        torch.save({'rng':_rng_state(device),'feature_matrix':operator.make_orthogonal_features(mixer.heads,tokens.shape[-1]//mixer.heads,int(options['rank']),int(options['feature_seed']),dtype=torch.float32),'source':source,'job':job,'training_ids':sample_ids},handle);handle.flush();os.fsync(handle.fileno())
    os.replace(temporary,rng_path);row['state_record']=_record(rng_path);_atomic_json(job['result_path'],row)
    return row

def _subprocess_case(root,job_path,timeout=180):
    """No fork after CUDA initialization; kill the entire fresh case process group on cutoff."""
    path=Path(root)/'experiments/17_fixed_kernel/benchmark.ipynb'
    launch="import nbformat,types,sys; p=sys.argv[1]; m=types.ModuleType('fmm17_case'); m.__file__=p; [exec(compile(c.source,p,'exec'),m.__dict__) for c in nbformat.read(p,4).cells if c.cell_type=='code' and 'ahr-export' in c.metadata.get('tags',[])]; m.run_compression_case(sys.argv[2],sys.argv[3])"
    log=Path(job_path).with_suffix('.log')
    with log.open('a')as handle:
        process=subprocess.Popen([sys.executable,'-u','-c',launch,str(path),str(root),str(job_path)],stdout=handle,stderr=subprocess.STDOUT,start_new_session=True)
        try:code=process.wait(timeout=max(.01,min(180.,float(timeout))))
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGTERM)
            try:process.wait(timeout=5)
            except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait(timeout=5)
            return {'status':'UNKNOWN_CASE_TIMEOUT','watchdog_seconds':min(180.,float(timeout)),'log':str(log),'no_silent_restart':True}
    if code!=0:return {'status':'UNKNOWN_CASE_FAILURE','exit_code':code,'log':str(log),'no_silent_restart':True}
    job=_read(job_path)
    if not Path(job['result_path']).exists():return {'status':'UNKNOWN_MISSING_CASE_RECEIPT','log':str(log)}
    return _read(job['result_path'])

def _checkpoint_folder(root,artifact):
    artifact=Path(artifact).resolve();parts=artifact.relative_to(Path(root).resolve()/'artifacts'/'notebook_17').parts
    if not parts:raise ValueError('Artifact must include a run ID')
    return Path(root)/'checkpoints'/parts[0]/'compression'

def _save_benchmark_checkpoint(folder,identity,completed,active_case=None):
    """Full resumable stage state; optimizer/scheduler/scaler are N/A for frozen operator cases."""
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);manifest_path=folder/'manifest.json'
    previous=_read(manifest_path)if manifest_path.exists()else None
    if previous and previous['identity']!=identity:raise ValueError('Benchmark checkpoint source/config mismatch; never restart')
    sequence=int(previous.get('sequence',0))+1 if previous else 1;path=folder/('stage-'+str(sequence).zfill(6)+'.pt');temporary=path.with_suffix('.tmp')
    payload={'schema':'fmm17-frozen-benchmark-v1','identity':identity,'completed':completed,'active_case':active_case,'position':len(completed),'rng':_rng_state(),
      'model':'Bound immutable source model checkpoint in each completed case; no model mutation','optimizer':None,'scheduler':None,'scaler':None,'config':identity,'metrics':completed,'sequence':sequence}
    with temporary.open('wb')as handle:torch.save(payload,handle);handle.flush();os.fsync(handle.fileno())
    os.replace(temporary,path);record=_record(path)
    # Best means the smallest observed worst-parameter fidelity error, not downstream accuracy.
    complete=[v for v in completed.values()if v.get('status')=='MEASURED'];score=min((v['errors']['max_relative_gradient_l2']for v in complete),default=None)
    best=record if previous is None or (score is not None and(previous.get('best_score')is None or score<previous['best_score']))else previous['best']
    latest_two=([record]+(previous.get('latest_two',[])if previous else[]))[:2]
    manifest={'schema':payload['schema'],'identity':identity,'sequence':sequence,'latest':record,'latest_two':latest_two,'best':best,'best_score':score if best==record else previous.get('best_score'),
      'resume_notebook':'notebooks/17_Fixed_Kernel_Feasibility_and_Learning.ipynb','resume_cell':'launch-fixed-kernel-study','completed_count':len(completed),'active_case':active_case,'single_writer_pid':os.getpid()}
    _atomic_json(manifest_path,manifest);keep={v['path']for v in latest_two+[best]}
    for old in folder.glob('stage-*.pt'):
        if str(old.resolve())not in keep:old.unlink()
    return manifest

def _restore_benchmark_checkpoint(folder,identity):
    path=Path(folder)/'manifest.json'
    if not path.exists():
        if Path(folder).exists()and any(Path(folder).glob('stage-*.pt')):raise ValueError('Orphan checkpoint without manifest; never silently restart')
        return {},None
    manifest=_read(path)
    if manifest['identity']!=identity:raise ValueError('Benchmark source/config changed; never silently restart')
    for record in manifest['latest_two']+[manifest['best']]:
        if _record(record['path'])!=record:raise ValueError('Invalid benchmark checkpoint checksum; never restart')
    payload=torch.load(manifest['latest']['path'],map_location='cpu',weights_only=False)
    if payload['identity']!=identity or payload['position']!=len(payload['completed']):raise ValueError('Checkpoint step/metric mismatch')
    for row in payload['completed'].values():
        if row.get('state_record')and _record(row['state_record']['path'])!=row['state_record']:raise ValueError('Case RNG/basis checkpoint hash mismatch')
    return payload['completed'],payload['active_case']


In [ ]:
def compression_benchmark(root,artifact,options=None,deadline=None,profile='full',handoff=None):
    root,artifact=Path(root),Path(artifact);artifact.mkdir(parents=True,exist_ok=True);common=_common(root)
    base={'feature_seed':170032,'local_radius':1,'spatial_mode':'disabled'};base.update(options or{})
    ranks=[32,64,128]if profile=='full'else[32];counts=[256,1024,4096]if profile=='full'else[25];layers=[0,3]if profile=='full'else[0]
    sources=common.source_hashes(root)if profile=='full'else {'benchmark_export':hashlib.sha256('\n'.join(c.source for c in nbformat.read(Path(__file__),4).cells if c.cell_type=='code'and'ahr-export'in c.metadata.get('tags',[])).encode()).hexdigest()}
    identity={'source_hashes':sources,'options':base,'ranks':ranks,'tokens':counts,'layers':layers,'batch':4 if profile=='full'else 2,'profile':profile,'deadline_unix':deadline,
      'output_fidelity_guard':.10,'all_shared_parameter_gradient_guard':.25,'selection':'Among ranks satisfying both fidelity guards in every learned-token case, choose the smallest median fixed/SDPA time ratio on N>=1024; tie by lower rank. No dev/test labels used.',
      'sampling_seed':2026171701,'layer_basis_policy':'feature_seed + layer index, identical to deployed FixedKernelMixer replacement','handoff':handoff,'requested_cases':len(ranks)*len(counts)*len(layers)}
    _immutable(artifact/'compression_registration.json',identity);folder=_checkpoint_folder(root,artifact);completed,active=_restore_benchmark_checkpoint(folder,identity)
    if active is not None:
        completed.setdefault(active,{'key':active,'status':'UNKNOWN_INTERRUPTED_CASE','reason':'Previous IN_FLIGHT stage checkpoint detected; no unreported case time forgiven'})
        _save_benchmark_checkpoint(folder,identity,completed,None)
    correctness_path=artifact/'implementation_correctness.json'
    if correctness_path.exists():correctness=_read(correctness_path)
    else:
        _deadline(deadline,10);correctness={'cpu':implementation_correctness(root,'cpu',32)}
        if profile=='full':
            job={'key':'gpu-finite-reference','device':'cuda:0','options':{**base,'rank':32},'correctness_only':True,'source_hashes':sources,'deadline':deadline,'result_path':str((artifact/'correctness_gpu.json').resolve())}
            job_path=artifact/'correctness_gpu_job.json';_immutable(job_path,job);correctness['gpu']=_subprocess_case(root,job_path,min(180.,float(deadline)-time.time())if deadline else 180.)
        correctness['passed']=all(v.get('passed',False)for v in correctness.values()if isinstance(v,dict));correctness['status']='PASS'if correctness['passed']else'FAIL';correctness['source_hashes']=sources
        _atomic_json(correctness_path,correctness)
    if correctness.get('source_hashes')!=sources:raise ValueError('Finite-reference correctness receipt source mismatch')
    if not correctness['passed']:
        report={'status':'NO_GO','passed':False,'reason':'Finite-kernel implementation correctness failed','identity':identity,'correctness':correctness,'rows':[],'eligible_ranks':[],'selected':None}
        _atomic_json(artifact/'compression_benchmark.json',report);return report
    # Parent releases all CUDA state before the fresh case subprocess checks exclusive ownership.
    gc.collect()
    if torch.cuda.is_initialized():torch.cuda.empty_cache()
    for rank in ranks:
        for N in counts:
            for layer in layers:
                key='r'+str(rank)+'-n'+str(N)+'-l'+str(layer)
                if key in completed:continue
                if deadline is not None and time.time()+5>=float(deadline):
                    completed[key]={'key':key,'rank':rank,'tokens':N,'layer':layer,'status':'UNKNOWN_GLOBAL_DEADLINE'};_save_benchmark_checkpoint(folder,identity,completed,None);continue
                job={'key':key,'root':str(root.resolve()),'artifact':str(artifact.resolve()),'tokens':N,'layer':layer,'options':{**base,'rank':rank,'feature_seed':int(base['feature_seed'])+int(layer)},'handoff':handoff,'device':'cuda:0'if profile=='full'else'cpu',
                  'software_fixture':profile!='full','deadline':deadline,'output_guard':.10,'gradient_guard':.25,'result_path':str((artifact/'cases'/key/'result.json').resolve())}
                job_path=artifact/'cases'/key/'job.json';_immutable(job_path,job);_save_benchmark_checkpoint(folder,identity,completed,key)
                timeout=min(180.,float(deadline)-time.time())if deadline else 180.;row=_subprocess_case(root,job_path,timeout)
                row.update(key=key,rank=rank,tokens=N,layer=layer);completed[key]=row;_save_benchmark_checkpoint(folder,identity,completed,None)
                _atomic_json(artifact/'compression_progress.json',{'status':'RUNNING','identity':identity,'completed':completed})
    rows=list(completed.values());measured=[r for r in rows if r['status']=='MEASURED'];complete=len(measured)==identity['requested_cases']
    eligible=[rank for rank in ranks if len([r for r in measured if r['rank']==rank])==len(counts)*len(layers)and all(r['fidelity_guard_pass']for r in measured if r['rank']==rank)]
    scored=[]
    for rank in eligible:
        observed=[r for r in measured if r['rank']==rank and(r['tokens']>=1024 or profile!='full')]
        scored.append((statistics.median(r['measurements']['fixed']['forward_backward_seconds']/r['measurements']['sdpa']['forward_backward_seconds']for r in observed),rank))
    selected={**base,'rank':min(scored)[1]}if scored and complete and profile=='full'else None
    report={'status':'PASS'if selected is not None else('NO_GO'if complete else'UNKNOWN'),'passed':selected is not None,'source_hashes':sources,'identity':identity,'correctness':correctness,'rows':rows,'eligible_ranks':eligible,'selected':selected,
      'profile':profile,'inference':'Training-only frozen operator screening; approximation fidelity is empirical and image quality is unmeasured. GPU/operator timings do not establish complete JEPA speed.',
      'finite_cost':'Fixed rank means O(B H N r d) summaries plus O(B H N 9 d) exact local correction; measured implementation cost still decides promotion.',
      'confirmation_eligible':False,'created_unix':time.time()}
    _atomic_json(artifact/'compression_progress.json',{'status':report['status'],'identity':identity,'completed':completed});_atomic_json(artifact/'compression_benchmark.json',report);return report


In [ ]:
def _cost_model(config,source_model,mode,selected):
    operator=_operator();current=copy.deepcopy(source_model)
    if mode=='fixed':operator.replace_encoder_mixers(current,{k:v for k,v in selected.items()if k in('rank','feature_seed','local_radius','spatial_mode')})
    return current.cpu()

def _move_optimizer(optimizer,device):
    for state in optimizer.state.values():
        for key,value in state.items():
            if torch.is_tensor(value):state[key]=value.to(device)

def _cost_save(folder,identity,models,states,samples,next_position,in_flight=None):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);path=folder/'manifest.json';old=_read(path)if path.exists()else None
    if old and old['identity']!=identity:raise ValueError('Cost checkpoint identity mismatch')
    sequence=(old['sequence']if old else 0)+1;destination=folder/('cost-'+str(sequence).zfill(6)+'.pt');temporary=destination.with_suffix('.tmp')
    payload={'schema':'fmm17-complete-jepa-cost/v1','identity':identity,'models':{k:m.state_dict()for k,m in models.items()},
      'optimizer':{k:s[0].state_dict()for k,s in states.items()},'scheduler':{k:s[1].state_dict()for k,s in states.items()},'scaler':None,'rng':_rng_state('cuda:0'),
      'sampler':{'policy':'Registered fixed128 TRAIN members; occurrence seed is pair index; same for both architectures','train_ids':identity['training_ids'],'next_complete_step':next_position},
      'global_step':next_position,'config':identity,'metrics':samples,'next_position':next_position,'in_flight':in_flight,'sequence':sequence}
    with temporary.open('wb')as handle:torch.save(payload,handle);handle.flush();os.fsync(handle.fileno())
    os.replace(temporary,destination);record=_record(destination);latest_two=([record]+(old.get('latest_two',[])if old else[]))[:2]
    # Best checkpoint records the most completed paired measurements; it is not a best accuracy model.
    best=record if in_flight is None else(old['best']if old else record)
    manifest={'schema':payload['schema'],'identity':identity,'sequence':sequence,'latest':record,'latest_two':latest_two,'best':best,'next_position':next_position,'in_flight':in_flight,'resume_notebook':'notebooks/17_Fixed_Kernel_Feasibility_and_Learning.ipynb','resume_cell':'launch-fixed-kernel-study','single_writer_pid':os.getpid()}
    _atomic_json(path,manifest);keep={r['path']for r in latest_two+[best]}
    for oldfile in folder.glob('cost-*.pt'):
        if str(oldfile.resolve())not in keep:oldfile.unlink()
    return manifest

def _cost_restore(folder,identity,models,states):
    path=Path(folder)/'manifest.json'
    if not path.exists():
        if Path(folder).exists()and any(Path(folder).glob('cost-*.pt')):raise ValueError('Cost checkpoint manifest missing; never silently restart')
        return [],0
    manifest=_read(path)
    if manifest['identity']!=identity:raise ValueError('Full-JEPA cost source/config changed; never restart')
    for record in manifest['latest_two']+[manifest['best']]:
        if _record(record['path'])!=record:raise ValueError('Full-JEPA cost checkpoint hash mismatch')
    payload=torch.load(manifest['latest']['path'],map_location='cpu',weights_only=False)
    if payload['identity']!=identity or payload['global_step']!=manifest['next_position']:raise ValueError('Cost resume step/metrics mismatch')
    if payload['in_flight']is not None:raise RuntimeError('UNKNOWN_IN_FLIGHT_COST: interrupted unsaved measurement; no silent restart or timing forgiveness')
    for mode,model in models.items():
        model.load_state_dict(payload['models'][mode]);states[mode][0].load_state_dict(payload['optimizer'][mode]);states[mode][1].load_state_dict(payload['scheduler'][mode])
    import random
    random.setstate(payload['rng']['python']);np.random.set_state(payload['rng']['numpy']);torch.set_rng_state(payload['rng']['torch_cpu']);torch.cuda.set_rng_state_all(payload['rng']['torch_cuda'])
    return payload['metrics'],int(payload['next_position'])

def full_jepa_speed_gate(root,artifact,selected,deadline=None,image_size=256,microbatch=16,repetitions=5,preflight=None,workers=None,profile='full'):
    """Alternating complete steps, real data, effective batch 128; discarded timing weights."""
    root,artifact=Path(root),Path(artifact);common=_common(root);pipeline=_pipeline(root);R17=_runner(root)
    if preflight is None and(artifact/'preflight_result.json').exists():preflight=_read(artifact/'preflight_result.json')
    if isinstance(preflight,dict)and'certificate'in preflight:
        if preflight.get('status')!='PASS'or not preflight.get('passed',False):raise ValueError('Discarded actual preflight wrapper did not pass')
        preflight=preflight['certificate']
    arms=preflight.get('arms',{})if isinstance(preflight,dict)else{}
    if not isinstance(preflight,dict)or preflight.get('status')!='PASS'or len(arms)<2 or not all(a.get('status')=='PASS'and a.get('bitwise_equal',False)for a in arms.values()):raise ValueError('Full-state exact 4 versus 2+2 checkpoint resume proof required before complete JEPA timing')
    if int(microbatch)!=16 or int(image_size)not in(256,512):raise ValueError('Registered speed gate uses physical batch16, effective128, image256or512')
    if preflight.get('binding',{}).get('source_hashes')!=common.source_hashes(root):raise ValueError('Preflight export sources differ from current complete-JEPA cost sources')
    if any(a.get('image_size')!=int(image_size)or a.get('microbatch')!=16 or a.get('accumulation_steps')!=8 or a.get('schedule_updates')!=4000 for a in arms.values()):raise ValueError('Preflight resolution, batch, accumulation or optimizer schedule differs')
    preflight_runner,_,_=common.modules(root)
    for entry in arms.values():
        for record in entry.get('checkpoints',{}).values():
            manifest=preflight_runner.validate_training_manifest(Path(record['directory']))
            if manifest['successful_updates']!=4 or preflight_runner.digest_file(Path(record['directory'])/manifest['latest'])!=record['sha256']:raise ValueError('Preflight checkpoint evidence changed')
    _deadline(deadline,30);common.environment({'device':'cuda:0'});exclusive=_exclusive_gpu('cuda:0');model,config,source=common.learned_source(root,None,variant='vit');model.cpu()
    config=copy.deepcopy(config);config.update(device='cuda:0',image_size=int(image_size));config['training'].update(microbatch=16,accumulation_steps=8,updates=4000,checkpoint_every=50)
    common.environment(config);bundle=R17.data_bundle(root,artifact/'cost_data',config)
    if preflight.get('binding',{}).get('training_ids_sha256')!=preflight_runner._hash(bundle['train_ids']):raise ValueError('Preflight training membership differs from complete-step timing')
    pipeline_receipt=pipeline.benchmark_input_pipeline(bundle,config,deadline=deadline)if workers is None else {'status':'PASS','selected_workers':int(workers),'source':'predeclared shared optimized pipeline receipt'}
    selected_workers=pipeline_receipt.get('selected_workers',pipeline_receipt.get('workers'))
    if selected_workers is None:raise ValueError('Pipeline optimization did not return selected_workers')
    verification=pipeline.verify_pipeline(bundle,config,workers=(int(selected_workers),),deadline=deadline);pipeline.install_input_pipeline(workers=int(selected_workers))
    if verification.get('status')!='PASS':raise ValueError('Optimized pipeline changed original batches/masks')
    from fmm_jepa.food101_data import make_batch
    from fmm_jepa.semantic_pilot import pretraining_forward,move_masks,make_optimizer
    runner,encoder,operator=common.modules(root);models={mode:_cost_model(config,model,mode,selected)for mode in('sdpa','fixed')};states={};policy=runner._strict_training_protocol(config)
    for mode,current in models.items():
        optimizer,scheduler,groups=make_optimizer(current,policy);states[mode]=(optimizer,scheduler,groups)
    identities=bundle['train_ids'][:128];identity={'source_hashes':common.source_hashes(root),'selected':selected,'source':source,'image_size':int(image_size),'physical_batch':16,'effective_batch':128,'dtype':'float32','repetitions':int(repetitions),'workers':int(selected_workers),'preflight':preflight,'deadline_unix':deadline,'training_ids':identities,'order':'Warmup pair then five pairs; SDPA/fixed order alternates by pair; identical occurrence seeds per pair.'}
    _immutable(artifact/'full_jepa_speed_registration.json',identity);folder=_checkpoint_folder(root,artifact).parent/('cost-'+str(image_size));samples,next_position=_cost_restore(folder,identity,models,states)
    order=[(repeat,mode)for repeat in range(int(repetitions)+1)for mode in(('sdpa','fixed')if repeat%2==0 else('fixed','sdpa'))]
    if next_position!=len(samples):raise ValueError('Cost resume telemetry position mismatch')
    for position,(repeat,mode)in enumerate(order):
        if position<next_position:continue
        _deadline(deadline,10);_exclusive_gpu('cuda:0');current=models[mode];optimizer,scheduler,groups=states[mode]
        # The inactive architecture lives on CPU, so the CUDA peak is the ordinary whole active JEPA allocation.
        _cost_save(folder,identity,models,states,samples,position,in_flight={'position':position,'pair':repeat,'mode':mode})
        current.to(config['device']);_move_optimizer(optimizer,config['device']);optimizer.zero_grad(set_to_none=True);observer=_ComputeObserver(config['device']).start()
        torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();started=time.perf_counter();data_wall=0.;events=[]
        for micro in range(8):
            _deadline(deadline,1);data_started=time.perf_counter();a=micro*16;b=(micro+1)*16;batch=make_batch(bundle['dataset'],identities[a:b],[repeat]*16,config,training=True)
            images=batch['images'].to(config['device']);masks=move_masks(batch['masks'],torch.device(config['device']));data_wall+=time.perf_counter()-data_started
            e0=torch.cuda.Event(enable_timing=True);e1=torch.cuda.Event(enable_timing=True);e0.record();output=pretraining_forward(current,{'images':images,'masks':masks});(output['loss']/8).backward();e1.record();events.append((e0,e1))
        grads=[p.grad for p in current.parameters()if p.requires_grad and p.grad is not None]
        if not grads or not all(bool(torch.isfinite(g).all())for g in grads)or not bool(torch.isfinite(output['loss'])):raise FloatingPointError('Nonfinite complete JEPA step')
        if any(p.grad is not None for p in current.target.parameters()):raise AssertionError('EMA teacher received gradients')
        e0=torch.cuda.Event(enable_timing=True);e1=torch.cuda.Event(enable_timing=True);e0.record();torch.nn.utils.clip_grad_norm_([p for p in current.parameters()if p.requires_grad],float(policy['gradient_clip']),error_if_nonfinite=True)
        optimizer.step();current.update_teacher(scheduler.ema_for(repeat+1));scheduler.step();e1.record();events.append((e0,e1));torch.cuda.synchronize();wall=time.perf_counter()-started;peak=torch.cuda.max_memory_allocated()
        sample={'mode':mode,'pair':repeat,'warmup':repeat==0,'complete_step_seconds':wall,'data_seconds':data_wall,'compute_cuda_seconds':sum(a.elapsed_time(b)/1000 for a,b in events),'peak_cuda_allocated_bytes':int(peak),'finite':True,'observed_compute_availability':observer.finish()}
        samples.append(sample);current.cpu();_move_optimizer(optimizer,'cpu');del output,grads,images,masks,batch;gc.collect();torch.cuda.empty_cache()
        _cost_save(folder,identity,models,states,samples,position+1,None);_atomic_json(artifact/'full_jepa_speed_progress.json',{'status':'RUNNING','identity':identity,'samples':samples})
    results={mode:[s for s in samples if s['mode']==mode and not s['warmup']]for mode in models}
    aggregate={mode:{'median_complete_step_seconds':statistics.median(s['complete_step_seconds']for s in values),'median_compute_cuda_seconds':statistics.median(s['compute_cuda_seconds']for s in values),'median_data_seconds':statistics.median(s['data_seconds']for s in values),'peak_cuda_allocated_bytes':max(s['peak_cuda_allocated_bytes']for s in values),'measurements':values}for mode,values in results.items()}
    speed=aggregate['sdpa']['median_complete_step_seconds']/aggregate['fixed']['median_complete_step_seconds'];memory=aggregate['fixed']['peak_cuda_allocated_bytes']<=aggregate['sdpa']['peak_cuda_allocated_bytes'];passed=speed>=1.10 and memory
    report={'status':'PASS'if passed else'NO_GO','passed':passed,'source_hashes':identity['source_hashes'],'identity':identity,'results':aggregate,'measured_end_to_end_speed_over_sdpa':speed,'required_speed':1.10,'memory_no_increase':memory,'pipeline':pipeline_receipt,'pipeline_verification':verification,'exclusive_gpu':exclusive,'image_size':int(image_size),'tokens':(int(image_size)//int(config['model']['patch_size']))**2,'selected':selected,'profile':profile,'source':source,'scope':'Complete real-data JEPA preprocessing/transfer/context/target/predictor/loss/backward/clipping/AdamW/EMA; equal physical/effective batch, FP32, masks and input pipeline; discarded weights; one warmup and alternating five measured pairs. Foreign idle CUDA contexts are allowed only under recorded no-foreign-SM observations before/during/after each measured step; host contention remains possible. Only active benchmark architecture is GPU-resident; complete CUDA allocated peak includes its model, optimizer, gradients and activations. CPU model swaps and measurement checkpoint I/O are outside per-step timing, shared equally.'}
    _atomic_json(artifact/'full_jepa_speed_gate.json',report);del models,states,model;gc.collect();torch.cuda.empty_cache();return report


In [ ]:
def _single_writer_stage(function):
    from functools import wraps
    @wraps(function)
    def guarded(root,artifact,*args,**kwargs):
        import fcntl
        folder=_checkpoint_folder(root,artifact).parent/'benchmark-writer';folder.mkdir(parents=True,exist_ok=True)
        with(folder/'writer.lock').open('a+')as handle:
            try:fcntl.flock(handle.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)
            except BlockingIOError:raise RuntimeError('Another benchmark writer owns this run; no concurrent mutation')
            handle.seek(0);handle.truncate();handle.write(str(os.getpid()));handle.flush()
            try:return function(root,artifact,*args,**kwargs)
            finally:
                for observer in list(_ComputeObserver._active):observer.abort()
                fcntl.flock(handle.fileno(),fcntl.LOCK_UN)
    return guarded

compression_benchmark=_single_writer_stage(compression_benchmark)
full_jepa_speed_gate=_single_writer_stage(full_jepa_speed_gate)
